# Mode B: Memetic NSGA-II

**NSGA-II + Local Search** - Applies local search to improve individuals after genetic operators.

## 1. Imports

In [1]:
from __future__ import annotations
import random, copy, time
import numpy as np
from pathlib import Path
from datetime import datetime

from deap import base, creator, tools
from schedule_engine.notebooks.core import (
    load_data, create_random_individual, course_aware_crossover, smart_mutation,
    create_evaluator, get_constraint_breakdown, setup_deap, get_best_individual, 
    EvolutionStats, print_constraint_details
)
from schedule_engine.notebooks.viz import plot_convergence, plot_constraint_breakdown, print_summary
from schedule_engine.notebooks.strategies import local_search_individual

print(" Imports successful")

 Imports successful


## 2. Mode B Configuration

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# GA Parameters - FIXED for proper exploration
POP_SIZE = 100          # Was 10 - way too small!
NGEN = 500              # Was 50000 - wasteful with stagnation
CXPB = 0.8              # Slightly lower crossover
MUTPB = 0.4             # Higher mutation to escape local optima
FITNESS_WEIGHTS = (-1.0, -0.01)  # Hard has 100x priority over soft

# MODE B: Local search parameters - REDUCED to preserve diversity
LOCAL_SEARCH_PROB = 0.1         # Was 0.2 - too aggressive
LOCAL_SEARCH_ITERATIONS = 5     # Was 10 - reduce intensity

# Paths
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/mode_b_memetic/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Mode B: pop={POP_SIZE}, ngen={NGEN}, LS_prob={LOCAL_SEARCH_PROB}")
print(f"   Key changes: Larger pop, fewer gens, higher mutation")

 Mode B: pop=10, ngen=50000, LS_prob=0.2


## 3. Load Data

In [3]:
data = load_data(
    data_dir=DATA_DIR,
    opening_time="10:00",
    closing_time="17:00",
    closed_days=["Saturday"],
)

print(f" {data.summary()}")

️  Non-schedulable courses filtered out

CE604: BCE5A, BCE5B, BCE5C, BCE5D, BCE5E, BCE5F (0 credits/LTP)

ENCE 256: BCE4A, BCE4B, BCE4C, BCE4D, BCE4E, BCE4F (0 credits/LTP)

ENIE 254: BIE4A, BIE4B (0 credits/LTP)

ME706: BME7A, BME7B (0 credits/LTP)

16 enrollments skipped (Survey Camp, Industrial Attachment, etc.)

 Courses: 668, Groups: 74, Instructors: 181, Rooms: 67, Quanta: 42


## 4. Test Components

In [4]:
# Test individual creation and evaluation
evaluate = create_evaluator(data)
test_ind = create_random_individual(data)
print(f" Individual: {len(test_ind)} genes")
print(f" Initial fitness: hard={evaluate(test_ind)[0]}")

# Test local search
improved_ind, improvement = local_search_individual(
    test_ind, data, evaluate, max_iterations=5
)
print(f" After LS: hard={evaluate(improved_ind)[0]} (improvement={improvement})")

 Individual: 527 genes
 Initial fitness: hard=2495.0
 After LS: hard=2490.0 (improvement=5.0)


## 5. Memetic NSGA-II Evolution (Mode B Specific)

In [ ]:
LOG_INTERVAL = 10  # Show detailed breakdown every N gens

def run_memetic_nsga2():
    """Run NSGA-II with memetic local search."""
    print(f" Memetic NSGA-II: pop={POP_SIZE}, ngen={NGEN}, LS_prob={LOCAL_SEARCH_PROB}")
    start = time.time()
    
    setup_deap(FITNESS_WEIGHTS)
    
    toolbox = base.Toolbox()
    toolbox.register("individual", lambda: creator.Individual(create_random_individual(data)))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", course_aware_crossover)
    toolbox.register("mutate", lambda ind: smart_mutation(ind, data))
    toolbox.register("select", tools.selNSGA2)
    
    pop = toolbox.population(n=POP_SIZE)
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)
    
    stats = EvolutionStats()
    
    for gen in range(NGEN):
        offspring = [copy.deepcopy(ind) for ind in toolbox.select(pop, len(pop))]
        
        # Crossover
        for i in range(0, len(offspring)-1, 2):
            if random.random() < CXPB:
                toolbox.mate(offspring[i], offspring[i+1])
                del offspring[i].fitness.values
                del offspring[i+1].fitness.values
        
        # Mutation
        for ind in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(ind)
                del ind.fitness.values
        
        # === MODE B: Local Search ===
        for ind in offspring:
            if random.random() < LOCAL_SEARCH_PROB:
                genes = list(ind)
                improved_genes, _ = local_search_individual(genes, data, evaluate, LOCAL_SEARCH_ITERATIONS)
                ind[:] = improved_genes
                del ind.fitness.values
        
        # Evaluate
        for ind in offspring:
            if not ind.fitness.valid:
                ind.fitness.values = toolbox.evaluate(ind)
        
        pop = toolbox.select(pop + offspring, POP_SIZE)
        
        # Stats
        hard_vals = [ind.fitness.values[0] for ind in pop]
        soft_vals = [ind.fitness.values[1] for ind in pop]
        stats.generations.append(gen)
        stats.min_hard.append(float(min(hard_vals)))
        stats.avg_hard.append(float(np.mean(hard_vals)))
        stats.max_hard.append(float(max(hard_vals)))
        stats.feasible_count.append(sum(1 for h in hard_vals if h == 0))
        stats.min_soft.append(float(min(soft_vals)))
        stats.avg_soft.append(float(np.mean(soft_vals)))
        
        # Show detailed breakdown every LOG_INTERVAL gens
        if gen % LOG_INTERVAL == 0 or gen == NGEN - 1:
            best_ind = min(pop, key=lambda ind: (ind.fitness.values[0], ind.fitness.values[1]))
            breakdown = get_constraint_breakdown(list(best_ind), data)
            hard_names = {'student_group_exclusivity', 'instructor_exclusivity', 'instructor_qualifications', 
                         'room_suitability', 'room_exclusivity', 'instructor_time_availability', 
                         'room_time_availability', 'course_completeness'}
            hard_bd = {k: v for k, v in breakdown.items() if k in hard_names}
            soft_bd = {k: v for k, v in breakdown.items() if k not in hard_names}
            print_constraint_details(hard_bd, soft_bd, gen)
    
    stats.elapsed_time = time.time() - start
    print(f" Done in {stats.elapsed_time:.1f}s")
    return pop, stats

final_pop, stats = run_memetic_nsga2()

 Memetic NSGA-II: pop=10, ngen=50000, LS_prob=0.2
  Gen   0:  Hard=2268  Soft=1388
         HARD: [course_comple=   0 | instructor_ex= 115 | instructor_qu= 390 | instructor_ti= 405 | room_exclusiv= 358 | room_suitabil= 218 | room_time_ava=   0 | student_group= 782]
         SOFT: [instructor_sc=  41 | paired_cohort=   0 | session_conti= 835 | student_lunch= 436 | student_sched=  76]
  Gen  10:  Hard=2087  Soft=1299
         HARD: [course_comple=   0 | instructor_ex= 156 | instructor_qu= 281 | instructor_ti= 401 | room_exclusiv= 312 | room_suitabil= 150 | room_time_ava=   0 | student_group= 787]
         SOFT: [instructor_sc=  27 | paired_cohort=   0 | session_conti= 920 | student_lunch= 272 | student_sched=  80]
  Gen  20:  Hard=1888  Soft=1224
         HARD: [course_comple=   0 | instructor_ex= 138 | instructor_qu= 187 | instructor_ti= 404 | room_exclusiv= 322 | room_suitabil= 102 | room_time_ava=   0 | student_group= 735]
         SOFT: [instructor_sc=  35 | paired_cohort=   0 | sess

## 6. Results & Visualization

In [ ]:
best = get_best_individual(final_pop)
breakdown = get_constraint_breakdown(best, data)

print_summary(final_pop, stats, breakdown)

plot_convergence(stats, OUTPUT_DIR / "mode_b_convergence.png", title_prefix="Mode B: ")
plot_constraint_breakdown(breakdown, OUTPUT_DIR / "mode_b_breakdown.png", title="Mode B: Constraint Violations")

## 7. Export Results

In [ ]:
from schedule_engine.notebooks.export import export_full_results

export_paths = export_full_results(
    population=final_pop,
    stats=stats,
    data=data,
    output_dir=OUTPUT_DIR,
    mode_name="mode_b_memetic",
)

print(f"\n All files saved to: {OUTPUT_DIR}")